# Load data

In [3]:
import pandas as pd

data_path = "../../../data/bpRNA/"
df = pd.read_csv(data_path + "bpRNA.csv")
rfam = pd.read_csv(data_path +"families_bpRNA-new/bprna-new_ID_to_Type_Rfam14_2.csv")

In [ ]:
# split bpRNA and bpRNA-new sequences into train and test sets
is_bprna = df["id"].str.startswith("bpRNA_")

df_bprna = df[is_bprna].copy() 
df_new = df[~is_bprna].copy() 

save_cols = ["fold_name","fold_number","partition","id"]
df = pd.DataFrame(columns=save_cols) 
df_bprna["partition"] = "train"
df_bprna["fold_name"] = "bprna"
df_bprna["fold_number"] = 0


df_new["partition"] = "test"
df_new["fold_name"] = "bprna-new"
df_new["fold_number"] = 0
df = pd.concat([df, df_bprna[save_cols], df_new[save_cols]], axis=0)
df.to_csv("splits/fam_folds_train_split.csv", index=False)

In [ ]:
# use the Rfam family information to create a test split 
# for the top 20 families in bpRNA-new

df_newf = rfam.query('id in @df_new.id')
 
folds = []

top_fams = df_newf["fam"].value_counts().head(20).index
train_ids = df_bprna["id"].to_numpy()

for fold_number, fam in enumerate(top_fams):
    test_ids = df_newf.loc[df_newf["fam"] == fam, "id"].to_numpy()

    folds.append(pd.DataFrame({
        "fold_name": fam,
        "fold_number": fold_number,
        "partition": "test",
        "id": test_ids,
    })) 

df_fam_split = pd.concat(folds, ignore_index=True)
df_fam_split.to_csv("splits/fam_folds_test_split.csv", index=False)